# Vocal Accessibility Layer -- Pipeline Walkthrough

Runs the full pipeline (ASR -> Speech Equalizer -> Normalizer -> Orchestrator -> Guardrail -> Skill -> Transparency) stage by stage on a sample stuttered, mixed-context utterance.

Uses the `mock` backends so this runs anywhere with no API key, model download, or microphone -- see `../README.md` for switching to real ASR/LLM backends (`local`/`ollama`/`event`) once available.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.foundation.asr import get_asr_backend
from src.foundation.equalizer import equalize
from src.foundation.normalizer import normalize
from src.agent.orchestrator import orchestrate
from src.agent.guardrail import check as guardrail_check
from src.agent.memory import Memory
from src.llm import get_llm_backend
from src.skills import get_skills
from src.transparency import TransparencyReport

llm = get_llm_backend()
asr = get_asr_backend()
memory = Memory(user_id='notebook-demo')
memory.seed_faq([
    {"question": "What is PAS 901?", "answer": "PAS 901:2025 is the Vocal Accessibility code of practice."},
])
print('Skills registered:', list(get_skills().keys()))

Skills registered: ['faq_lookup', 'general_help', 'schedule_reminder']


## Stage 1: ASR
In `mock` mode this reads a sibling `.txt` transcript instead of running real Whisper -- see `data/sample_audio/example1.txt`.

In [2]:
asr_result = asr.transcribe('../data/sample_audio/example1.wav')
print('Raw transcript:', asr_result.text)
print('ASR confidence:', asr_result.confidence)

Raw transcript: M-m-m-my na-name i-is So-soumesh, s-schedule a m-meeting at 5 PM
ASR confidence: 0.75


## Stage 2: Speech Equalizer
Cleans disfluent/stuttered speech into a fluent sentence, preserving meaning and named entities.

In [3]:
eq_result = equalize(asr_result.text, llm)
print('Clean text:', eq_result.clean_text)
print('Corrections made:', eq_result.corrections_made)

Clean text: M-my na-name i-is So-soumesh, s-schedule a m-meeting at 5 PM
Corrections made: True


## Stage 3: Normalizer
Extracts structured intent/entities and detects code-switched languages, rather than doing literal word-for-word translation.

In [4]:
norm_result = normalize(eq_result.clean_text, llm)
print('Intent:', norm_result.intent)
print('Entities:', norm_result.entities)
print('Languages detected:', norm_result.languages_detected)

Intent: unknown
Entities: {}
Languages detected: ['en']


## Stage 4: Orchestrator
Routes to the best-matching registered skill (or asks a clarifying question instead of failing) -- this is the extensibility point: a new match-day use case is a new file in `src/skills/`, nothing here changes.

In [5]:
orch_result = orchestrate(norm_result.intent, norm_result.entities, norm_result.clean_text, llm)
print('Routed skill:', orch_result.skill_name)
print('Params:', orch_result.params)
print('Clarifying question (if any):', orch_result.clarify)

Routed skill: schedule_reminder
Params: {}
Clarifying question (if any): None


## Stage 5: Guardrail
Blocks the action and asks for confirmation instead of confidently executing on low-confidence ASR -- this is the safety story: a system that mishears a disfluent user shouldn't silently act on the wrong thing.

In [6]:
skill_output = None
g_result = None
if orch_result.skill_name:
    skill = get_skills()[orch_result.skill_name]
    g_result = guardrail_check(asr_result.confidence, f'{skill.name}({orch_result.params})', llm)
    print('Guardrail approved:', g_result.approved, '-', g_result.reason)
    if g_result.approved:
        skill_result = skill.run(orch_result.params, memory)
        skill_output = skill_result.output
        print('Skill output:', skill_output)

Guardrail approved: True - confidence above threshold
Skill output: Scheduled 'your reminder' for an unspecified time.


## Stage 6: Transparency
What the user actually sees -- confidence, what was corrected, what languages were detected, what action was taken.

In [7]:
report = TransparencyReport(
    asr_confidence=asr_result.confidence,
    corrections_made=eq_result.corrections_made,
    languages_detected=norm_result.languages_detected,
    skill_invoked=orch_result.skill_name or '',
    guardrail_approved=g_result.approved if g_result else True,
    guardrail_reason=g_result.reason if g_result else '',
)
for badge in report.badges():
    print('-', badge)

- Speech recognized by AI (confidence: 75%)
- Disfluency corrected
- Action: schedule_reminder
- Verified


## Extensibility proof
See `tests/test_pipeline_smoke.py::test_new_skill_can_be_added_without_touching_orchestrator` for an automated test that adds a brand-new skill file at runtime and confirms it's picked up with zero changes to the orchestrator, guardrail, or foundation layer.